In [49]:
import torch
import torch.nn as nn

import numpy as np
import pandas as pd

In [23]:
import math

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model) # Lookup table

    def forward(self, x):
        x = self.token_embedding(x) # (B, 1, seq_len) --> (B, seq_len, d_model)
        x = x * math.sqrt(self.d_model) #  scale the embeddings before adding positional encoding to keep the two on a similar magnitude
        return x

In [ ]:
class PsitionEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        position_encoding = torch.zeros(seq_len, d_model)
        
        pos = torch.arange(0, seq_len).unsqueeze(-1)
        i = torch.arange(0, d_model, 2).unsqueeze(0)
        position_encoding[:, 0::2] = torch.sin(pos/10000**(i/d_model))
        position_encoding[:, 1::2] = torch.cos(pos/10000**(i/d_model))
        
        # for pos in range(position_encoding.shape[0]):
        #     for i in range(position_encoding.shape[1]):
        #         if i % 2 == 0:
        #             position_encoding[pos, i] = np.sin(pos/10000**(i/d_model))
        #         else:
        #             position_encoding[pos, i] = np.cos(pos/10000**((i-1)/d_model))
        
        # need to register as buffer to let Pytorch not train the weights
        self.register_buffer("position_encoding", position_encoding) # (seq_len, d_model)

    def forward(self, x):
        x = x + self.position_encoding.unsqueeze(0) # (B, seq_len, d_model) + (1, seq_len, d_model) --> Pytorch broadcasting takes care of batch dimension
        return x

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model should be divisible by h"
        self.d_k = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    @staticmethod
    def attention(q, k, v, mask=None):
        d_k = q.shape[-1]
        attention_scores = q @ k.transpose(-2, -1) / math.sqrt(d_k) # skippin B, h (seq_len, d_k) @ (d_k, seq_len) --> (seq_len, seq_len) or (B, h, seq_len, seq_len)
        if mask is not None:
            # apply mask
            pass
        attention_scores = attention_scores.softmax(dim=-1)
        x = attention_scores @ v # (seq_len, seq_len) @ (seq_len, d_k) --> (seq_len, d_k) or (B, h, seq_len, d_k)
        return x, attention_scores

    def forward(self, x): # x --> (B, seq_len, d_model)
        q = self.w_q(x) # --> (B, seq_len, d_model)
        k = self.w_k(x) # --> (B, seq_len, d_model)
        v = self.w_v(x) # --> (B, seq_len, d_model)

        q = q.view(q.shape[0], q.shape[1], self.h, self.d_k).transpose(1, 2) # (B, h, seq_len, d_k)
        k = k.view(k.shape[0], k.shape[1], self.h, self.d_k).transpose(1, 2) # (B, h, seq_len, d_k)
        v = v.view(v.shape[0], v.shape[1], self.h, self.d_k).transpose(1, 2) # (B, h, seq_len, d_k)

        x, self.attention_scores = MultiHeadAttention.attention(q, k, v) # (B, h, seq_len, d_k)
        x = x.transpose(1, 2).contiguou() # (B, seq_len, h, d_k)
        x = x.view(x.shape[0], x.shape[1], int(self.h * self.d_k)) # (B, seq_len, d_model)
        
        x = self.w_o(x)
        return x

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.l1 = nn.Linear(d_model, 4*d_model)
        self.ReLU = nn.ReLU()
        self.l2 = nn.Linear(4*d_model, d_model)

    def forward(self, x):
        x = self.l1(x) # (B, seq_len, d_model) --> (B, seq_len, 4*d_model)
        x = self.ReLU(x)
        x = self.l2(x) # (B, seq_len, 4*d_model) --> (B, seq_len, d_model)
        return x

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(d_model)) # scale each column by one value (alpha)
        self.bias = nn.Parameter(torch.zeros(d_model)) # shift each column by one value (bias)

    def forward(self, x):
        # standardize values for each row; x (B, seq_len, d_model)
        x = (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True) + self.eps)
        # for row_id in range(x.shape[1]):
        #     row = x[:, row_id, :]
        #     row_mean = row.mean()
        #     row_var = row.std() ** 2
        #     x[:, row_id, :] = (row - row_mean) / math.sqrt(row_var + eps)

        x = x * self.alpha + self.bias
        return x

In [ ]:
class ResidualConnection(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.norm = LayerNorm(d_model, eps)

    def forward(self, x, sublayer: MultiHeadAttention | FeedForward):
        x = x + sublayer(self.norm(x))
        return x

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        

    def forward(self, x):

        return x